In [2]:
import numpy as np


def cosine_similarity(a,b):
    a = np.array(a)
    b = np.array(b)

    return float((a @ b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [3]:
cat = [0.8,0.6,0.3]
dog = [0.75,0.65,0.35]
car = [-0.5,0.2,0.9]

In [4]:
cat_dog = cosine_similarity(cat, dog)    # similarity Search
cat_car = cosine_similarity(cat, car)

print(cat_dog, cat_car)

0.9966186334192181 -0.00913251529954447


In [5]:
%pip install langchain-huggingface
%pip  install sentence-transformers


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


#### langchain-huggingface - Real embeddings


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLm-L6-v2"
)

e:\Rag_Learning\ragEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10902.27it/s]


In [8]:
query = "I love python programming and GenAi "
vectors = embeddings.embed_query(query)

In [9]:
len(vectors)

384

In [10]:
documents = [

    "Python is great for machine learning.",
    "Java is used in enterprise software",
    "Dogs are loyal and friendly animals",
    'The stock market dropped today',
    "Neural Networks learn from data ",
    "I enjoy studying programming language"

]

In [11]:
doc_vectors = embeddings.embed_documents(documents)

In [12]:
len(doc_vectors)

6

In [13]:
len(doc_vectors[0])

384

#### Similarity Search

In [14]:
query = "Best language for Ai"

q_vect= embeddings.embed_query(query)

In [15]:
query = "best animals for home"

q_vect= embeddings.embed_query(query)

In [16]:
scores = []

for i, vec in enumerate(doc_vectors):
    scores.append({"document":documents[i], "score":cosine_similarity(q_vect,vec)})

scores.sort(key=lambda ele: ele["score"], reverse=True)


for chunk in scores[:2]:
    print(chunk["document"])


Dogs are loyal and friendly animals
Python is great for machine learning.


In [17]:
scores

[{'document': 'Dogs are loyal and friendly animals',
  'score': 0.4261642108423156},
 {'document': 'Python is great for machine learning.',
  'score': 0.1523886615020637},
 {'document': 'I enjoy studying programming language',
  'score': 0.051401922018496665},
 {'document': 'The stock market dropped today', 'score': 0.01612202774486246},
 {'document': 'Neural Networks learn from data ',
  'score': -0.03543250021009157},
 {'document': 'Java is used in enterprise software',
  'score': -0.049226471530807225}]

#### With Real Vector Store

In [18]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_texts(texts=documents,embedding=embeddings)

In [19]:
query = "Best language for Ai"

In [20]:
chunks = vector_store.similarity_search(query,k=2)

In [21]:
chunks = vector_store.similarity_search_with_score(query,k=4)


In [22]:
chunks

[(Document(id='d9f5fea0-fbf1-4c3c-b4e8-15b693fb9e76', metadata={}, page_content='Python is great for machine learning.'),
  0.4660175646333374),
 (Document(id='36995e73-db79-46c1-8189-7e264df05c00', metadata={}, page_content='I enjoy studying programming language'),
  0.43532876774151935),
 (Document(id='3db2d6db-2b16-4d62-9b64-5a602d1178fe', metadata={}, page_content='Neural Networks learn from data '),
  0.288412849362921),
 (Document(id='d373ab60-20e8-4efc-844d-9506477b473e', metadata={}, page_content='Java is used in enterprise software'),
  0.22587463101410155)]

#### Cohere Embeddings

In [23]:
%pip install langchain-cohere


  Attempting uninstall: cohere

    Found existing installation: cohere 7.2.0

    Uninstalling cohere-7.2.0:

      Successfully uninstalled cohere-7.2.0

   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1/3 [cohere]
   ------------- -------------------------- 1


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
from langchain_cohere import CohereEmbeddings

from langchain_community.document_loaders import PyMuPDFLoader

from langchain_core.vectorstores import InMemoryVectorStore
from dotenv import load_dotenv
import os

In [32]:
os.getenv("COHERE_API_KEY")

'jIyXkmlaoJWZoAnDtlS5xIrEnLtHMsu6tRlu0aYR'

In [27]:
docs = PyMuPDFLoader(file_path="../data/Computer-Network-Notes.pdf")
docs = docs.load()

In [31]:
embeddings = CohereEmbeddings(model="embed-v4.0",cohere_api_key=os.getenv("COHERE_API_KEY"))

In [33]:
vector_store = InMemoryVectorStore.from_documents(documents=docs, embedding=embeddings)

In [34]:
res = embeddings.embed_query("Best Language for GenAi ")


In [35]:
len(res)

1536

In [40]:
records = vector_store.similarity_search("What is the tcp")

In [41]:
len(records)


4

In [42]:
print(records[0].page_content)

86 
 
 NTP – NTP stands for Network Time Protocol. It is used to synchronize the clocks on 
our computer to one standard time source. It is very useful in situations like bank 
transactions. Assume the following situation without the presence of NTP. Suppose you 
carry out a transaction, where your computer reads the time at 2:30 PM while the server 
records it at 2:28 PM. The server can crash very badly if it’s out of sync. 
 
Features 
 TCP is connection oriented. TCP requires that connection between two remote points 
be established before sending actual data. 
 TCP provides error-checking and recovery mechanism. 
 TCP provides end-to-end communication. 
 TCP provides flow control and quality of service. 
 TCP operates in Client/Server point-to-point mode. 
 
RPC 
 A remote procedure call is an interprocess communication technique that is used for 
client-server based applications. It is also known as a subroutine call or a function call. 
 A client has a request message tha